In [1]:
from utils.spark_session import get_sparkSession 

In [2]:
spark = get_sparkSession("data init - Load dim_currency")

In [3]:
from delta import DeltaTable

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
row_count = spark.read.table("gold.dim_currency").count()

if row_count ==0:
    spark.sql(
    """INSERT INTO gold.dim_currency (currency_sk,
                                     currency_code,
                                     currency_name,
                                     currency_country,
                                     is_deleted,
                                     created_on,
                                     created_by,
                                     modified_on,
                                     modified_by
                                    )
         VALUES
           (1,'USD','US Dollar','US',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (2,'INR','Indian Rupee','IN',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (3,'CAD','Canadian Dollar','CA',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (4,'JPY','Japanese Yen','JP',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (5,'KRW','South Korean Won','KR',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (6,'MXN','Mexican Peso','MX',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT')
    """)
    print("SPARK-APP: Rows Inserted")
else:
    print("SPARK-APP: Rows already exists")

SPARK-APP: Rows Inserted


In [6]:
spark.read.table("gold.dim_currency").show()

+-----------+-------------+----------------+----------------+----------+--------------------+----------+--------------------+-----------+
|currency_sk|currency_code|   currency_name|currency_country|is_deleted|          created_on|created_by|         modified_on|modified_by|
+-----------+-------------+----------------+----------------+----------+--------------------+----------+--------------------+-----------+
|          5|          KRW|South Korean Won|              KR|     false|2026-01-28 23:36:...| DATA-INIT|2026-01-28 23:36:...|  DATA_INIT|
|          3|          CAD| Canadian Dollar|              CA|     false|2026-01-28 23:36:...| DATA-INIT|2026-01-28 23:36:...|  DATA_INIT|
|          6|          MXN|    Mexican Peso|              MX|     false|2026-01-28 23:36:...| DATA-INIT|2026-01-28 23:36:...|  DATA_INIT|
|          4|          JPY|    Japanese Yen|              JP|     false|2026-01-28 23:36:...| DATA-INIT|2026-01-28 23:36:...|  DATA_INIT|
|          2|          INR|    Ind

In [7]:
dt = DeltaTable.forName(spark,"gold.dim_currency")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|1      |6            |
+-------+-------------+



In [9]:
spark.sql("describe detail gold.dim_currency").show(truncate = False)

+------+------------------------------------+-------------------------------+-----------+-------------------------------------------------------------+----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name                           |description|location                                                     |createdAt             |lastModified       |partitionColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+-------------------------------+-----------+-------------------------------------------------------------+----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|delta |6d352e9f-3c78-4f4e-934e-cd2985f65b19|spark_catalog.gold.dim_currency|null       |s3

In [ ]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, "gold.dim_currency")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

In [10]:
spark.stop()